# 03 - Abstract Parsing & Chunking

Parses raw PubMed abstract files into structured sections (Background, Methods, Results, Conclusions),
then chunks them into LangChain Documents with label-aware formatting and rich metadata.

**Input:** `data/raw/{pubid}.txt` files  
**Output:** LangChain Documents ready for vector DB ingestion, chunk statistics

In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
from collections import Counter
from src.data_loader import load_pubmedqa, build_mesh_lookup
from src.abstract_parser import parse_pubmed_file
from src.chunking import create_documents
import config

## Test Parsing on Sample Abstracts

In [ ]:
abstracts_dir = config.DATA_RAW_DIR
sample_files = sorted(f for f in os.listdir(abstracts_dir) if f.endswith(".txt"))[:3]

for filename in sample_files:
    filepath = os.path.join(abstracts_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        raw_text = f.read()

    parsed = parse_pubmed_file(raw_text)
    print(f"--- {filename} ---")
    print(f"  PMID: {parsed['pmid']}")
    print(f"  Title: {parsed['title'][:80]}...")
    print(f"  Sections: {[s['label'] for s in parsed['sections']]}")
    print(f"  Year: {parsed['publication_year']}")
    print()

## Create Document Chunks

In [ ]:
df = load_pubmedqa()
mesh_lookup = build_mesh_lookup(df)

documents, ids = create_documents(str(config.DATA_RAW_DIR), mesh_lookup)
print(f"\nTotal documents: {len(documents)}")
print(f"Total unique IDs: {len(set(ids))}")

## Chunk Statistics

In [ ]:
chunks_per_pub = Counter(d.metadata["pubid"] for d in documents)
print(f"Unique abstracts chunked: {len(chunks_per_pub)}")
print(f"Mean chunks per abstract: {sum(chunks_per_pub.values()) / len(chunks_per_pub):.3f}")
print(f"P75 chunks per abstract: {sorted(chunks_per_pub.values())[int(len(chunks_per_pub) * 0.75)]}")

section_dist = Counter(d.metadata["abstract_section"] for d in documents)
print(f"\nSection distribution:")
for section, count in section_dist.most_common():
    print(f"  {section}: {count}")

## Inspect Sample Chunks

In [ ]:
for doc in documents[:5]:
    print(f"ID: {doc.id}")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Metadata: section={doc.metadata['abstract_section']}, pubid={doc.metadata['pubid']}")
    print()

## Export Chunks for Inspection

In [ ]:
chunk_df = pd.DataFrame([
    {"chunk_id": doc.id, "chunk_text": doc.page_content, **doc.metadata}
    for doc in documents
])
chunk_df.to_csv(config.DATA_CHUNKS_DIR / "all_chunks.csv", index=False)
print(f"Exported {len(chunk_df)} chunks to data/chunks/all_chunks.csv")